In [ ]:
#@title Imports
import tensorflow as tf
from tensorflow.keras.models import load_model
from tkinter import *
import os
import librosa
import numpy as np
from tkinter import filedialog
from tkinter import messagebox

In [ ]:
#@title Load the model

# load the saved trained model
model_path = "C:\Machine_learning_Project_Data\IA\Trained model 2"
model = load_model(model_path)

# classes: a list of all the genres
classes = ['Classical', 'Electronic', 'Experimental', 'Folk', 'Hip-Hop', 'Instrumental', 'Jazz', 'Old-Time  Historic', 'Pop', 'Punk', 'Rock']

In [ ]:
#@title define commands

# input: the path of a given audio file
# output: the classified genre label
def classify_music(file_path):
  #kaiser_fast is a technique used for faster extraction
  #load the given path of the wav file: X - numpy array of the values of the amplitude for small time segments, sample_rate = the sample rate of the wav file
  X, sample_rate = librosa.load(file_path, res_type='kaiser_fast')
  # create a one dimensional array using X and sample_rate using short fourier transform and convertion to a logaritmic scale and decibels from amplitude
  mel =np.mean(librosa.feature.melspectrogram(y=X, sr=sample_rate).T,axis=0)
  # reshape the data into suitable shapes for the CNN model feed
  Sound=mel.reshape(1,16, 8,1)
  Sound.shape
  # pred1: the prediction of the model in onehotencode
  # pred2: the index of the prediction
  # genre: the classified genre
  pred1= model.predict(Sound)
  pred2 = np.argmax(pred1)
  genre = classes[pred2]
  return genre

# input: a certain path of an audio file
# output: activate the windows default media player and play the selected audio file
def play_wav_file(file_path):
    os.system('start "" "' + file_path + '"')


# output: open a dialog that lets the user browse all the file in the local machine
def browse_for_directory():
    directory = filedialog.askdirectory()
    directory_path.set(directory)

# output: summon the play_wav_file function and play the selected file from the scrollbox
def play_selected_file():
    file_path = file_listbox.get(file_listbox.curselection())
    play_wav_file(file_path)

# output: set the genre_label variable as the classified genre from the classify_music function of the selected file from the scrollbox
def classify_selected_file():
    file_path = file_listbox.get(file_listbox.curselection())
    genre = classify_music(file_path)
    genre_label.set(genre)

# Define the function to update the listbox with the files in the selected directory
def update_file_listbox():
    directory = directory_path.get()
    file_listbox.delete(0,END)
    for file_name in os.listdir(directory):
        if file_name.endswith('.wav'):
            file_listbox.insert(END, os.path.join(directory, file_name))

In [ ]:
#@title Creating the gui itself

# create the window, set the title and the dimensions of the window
root = Tk()
root.title('Music Genre Classifier')
root.geometry('1000x500')

# create a label at the top of the window
directory_label = Label(root, text='Select directory containing WAV files:')
directory_label.pack(pady=(20, 10))

# create a directory_path variable for later assigning it a value
directory_path = StringVar()

#create a button for browsing all the directories on the local machine below the firest label
browse_button = Button(root, text='Browse...', command=browse_for_directory)
browse_button.pack(pady=(0, 10))

# Create a label to display the selected directory path
directory_path_label = Label(root, textvariable=directory_path, font=('Helvetica', 12))
directory_path_label.pack()

# create a listbox that will display all the files from the selected directory
file_listbox = Listbox(root, font=('Helvetica', 12), height=10)
file_listbox.pack(pady=(20, 10), padx=20, fill=X)

#Create a scrollbar for the file listbox
file_listbox_scrollbar = Scrollbar(root)
file_listbox_scrollbar.pack(side=RIGHT, fill=Y)
file_listbox.config(yscrollcommand=file_listbox_scrollbar.set)
file_listbox_scrollbar.config(command=file_listbox.yview)

# Create a frame to hold the play button and genre label
bottom_frame = Frame(root)
bottom_frame.pack(side=BOTTOM, pady=20)

# Create a button to play the selected file
play_button = Button(bottom_frame, text='Play', font=('Helvetica', 12), command=play_selected_file)
play_button.pack(side=LEFT, padx=(20, 10))

# Create a button to classify the selected file
classify_button = Button(root, text='Classify Genre', command=classify_selected_file, font=('Helvetica', 12))
classify_button.pack(pady=(10, 0))

# Create a label to display the predicted genre
genre_label = StringVar()
genre_result_label = Label(root, textvariable=genre_label, font=('Helvetica', 14, 'bold'), fg='blue')
genre_result_label.pack(pady=(20, 0))

# Create a button to close the window
quit_button = Button(bottom_frame, text='Quit', font=('Helvetica', 12), command=root.destroy)
quit_button.pack(side=RIGHT, padx=(10, 20))

# Bind the file listbox to the directory path variable, so it updates when the path changes
directory_path.trace('w', lambda name, index, mode, sv=directory_path: update_file_listbox())

# create a loop so the GUI will keep running
root.mainloop()